In [1]:
import spikeinterface.full as si
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import pandas as pd
from scipy.interpolate import interp1d
from scipy.signal import medfilt
import os, sys
import shutil
from pprint import pprint
import time as time
import json

root_folder = Path(r"I:\Data\raw")


def ttl_from_analog(signal, fs, hysteresis=0.1, filt_kernel=5):
    """
    Analog to digital TTL conversion with hysteresis.
    Uses proper forward-fill to maintain binary state between transitions.
    """
    signal_f = medfilt(signal, kernel_size=filt_kernel)

    v_low  = np.percentile(signal_f, 5)
    v_high = np.percentile(signal_f, 95)
    v_lo   = v_low  + hysteresis * (v_high - v_low)
    v_hi   = v_high - hysteresis * (v_high - v_low)

    above_hi = signal_f > v_hi
    below_lo = signal_f < v_lo

    hi_crossings = np.where(np.diff(above_hi.astype(np.int8)) ==  1)[0] + 1
    lo_crossings = np.where(np.diff(below_lo.astype(np.int8)) ==  1)[0] + 1

    # Sparse change array, forward-filled via index propagation
    changes = np.full(len(signal_f), np.nan)
    changes[0]            = 1.0 if signal_f[0] > v_hi else 0.0
    changes[hi_crossings] = 1.0
    changes[lo_crossings] = 0.0

    mask      = ~np.isnan(changes)
    last_seen = np.where(mask, np.arange(len(changes)), 0)
    np.maximum.accumulate(last_seen, out=last_seen)
    digital   = changes[last_seen].astype(np.int8)

    d           = np.diff(digital)
    rising_idx  = np.where(d ==  1)[0] + 1
    falling_idx = np.where(d == -1)[0] + 1

    return digital, rising_idx, falling_idx


def quadrature_speed_direction(sigA, sigB, fs, pulses_per_rev=900,
                               hysteresis=0.1, max_gap_s=0.020):
    """
    x4 quadrature decoder.

    Uses all transitions on both channels (4 x pulses_per_rev events/rev).
    Inserts zero-speed sentinels exactly max_gap_s after each long inter-edge
    gap, so stopped periods and brief movements both resolve correctly.

    Parameters
    ----------
    sigA, sigB      : raw analog encoder channels
    fs              : sampling frequency (Hz)
    pulses_per_rev  : encoder lines per revolution (900 for H5-900-NE-S)
    hysteresis      : fraction of signal range used as hysteresis band
    max_gap_s       : inter-edge gap (s) above which speed is set to 0;
                      ~20 ms corresponds to ~1 cm/s at the mouse's position
                      on a 6 cm radius wheel
    """
    sig_len   = len(sigA)
    time_full = np.arange(sig_len) / fs

    digA, rising_A, falling_A = ttl_from_analog(sigA, fs, hysteresis)
    digB, rising_B, falling_B = ttl_from_analog(sigB, fs, hysteresis)

    # --- x4 decoding: direction logic for each edge type ---
    #   A rises  : forward if B=0, backward if B=1
    #   A falls  : forward if B=1, backward if B=0
    #   B rises  : forward if A=1, backward if A=0
    #   B falls  : forward if A=0, backward if A=1
    edges = np.concatenate([rising_A,  falling_A,
                             rising_B,  falling_B])
    dirs  = np.concatenate([
        np.where(digB[rising_A]  == 0,  1, -1),
        np.where(digB[falling_A] == 1,  1, -1),
        np.where(digA[rising_B]  == 1,  1, -1),
        np.where(digA[falling_B] == 0,  1, -1),
    ])

    if len(edges) < 2:
        return time_full * 1000, np.zeros(sig_len), np.zeros(sig_len)

    sort_idx = np.argsort(edges, kind='stable')
    edges    = edges[sort_idx]
    dirs     = dirs[sort_idx]

    t_edges      = edges / fs
    deg_per_edge = 360.0 / (pulses_per_rev * 4)

    dt          = np.diff(t_edges)
    speed_edges = (deg_per_edge * dirs[:-1]) / dt  # direction at interval start
    t_mid       = (t_edges[1:] + t_edges[:-1]) / 2

    # --- Zero-speed sentinels ---
    # Placed exactly max_gap_s after the last edge before each long gap,
    # so brief fast movements drop to zero promptly rather than at gap midpoint.
    long_gaps   = np.where(dt > max_gap_s)[0]
    t_zeros     = t_edges[long_gaps] + max_gap_s
    speed_zeros = np.zeros(len(long_gaps))
    dir_zeros   = np.zeros(len(long_gaps))

    t_mid_aug = np.concatenate([t_mid,       t_zeros])
    speed_aug = np.concatenate([speed_edges, speed_zeros])
    dir_aug   = np.concatenate([dirs[:-1],   dir_zeros])

    sort      = np.argsort(t_mid_aug, kind='stable')
    t_mid_aug = t_mid_aug[sort]
    speed_aug = speed_aug[sort]
    dir_aug   = dir_aug[sort]

    f_speed = interp1d(t_mid_aug, speed_aug, kind='previous',
                       bounds_error=False, fill_value=0)
    f_dir   = interp1d(t_mid_aug, dir_aug,   kind='previous',
                       bounds_error=False, fill_value=0)

    return time_full * 1000, f_speed(time_full), f_dir(time_full)


# ---------------------------------------------------------------------------
# Main loop
# ---------------------------------------------------------------------------

nidq_files = list(root_folder.rglob("*nidq.bin"))
print(f"Found {len(nidq_files)} NIDQ files to process.")

for bin_file in nidq_files:
    basefolder = bin_file.parent
    print(f"\nProcessing: {basefolder.name}")

    meta_path  = basefolder / 'Meta'
    plots_path = basefolder / 'plots'
    meta_path.mkdir(parents=True, exist_ok=True)
    plots_path.mkdir(parents=True, exist_ok=True)

    try:
        event = si.read_spikeglx(str(basefolder), stream_id='nidq',
                                  load_sync_channel=False)
        sf = event.get_sampling_frequency()

        sigA = event.get_traces(channel_ids=[event.get_channel_ids()[6]]).squeeze()
        sigB = event.get_traces(channel_ids=[event.get_channel_ids()[5]]).squeeze()

        ts_ms, speed, direction = quadrature_speed_direction(
            sigA, sigB, sf,
            pulses_per_rev=900,
            hysteresis=0.1,
            max_gap_s=0.020
        )

        df = pd.DataFrame({
            'time_ms'  : ts_ms,
            'speed'    : speed,
            'direction': direction
        })
        save_path = meta_path / 'speed.csv'
        df.to_csv(save_path, index=False)
        print(f"  Saved CSV : {save_path}")

        # --- Plot: stride-decimate to preserve spike shapes ---
        num_downsample_points = 5000
        if len(speed) > num_downsample_points:
            stride    = len(speed) // num_downsample_points
            speed_ds  = speed[::stride]
            time_s_ds = ts_ms[::stride] / 1000
        else:
            speed_ds  = speed
            time_s_ds = ts_ms / 1000

        plt.figure(figsize=(10, 4))
        plt.plot(time_s_ds, speed_ds, color='crimson', linewidth=1.5, alpha=0.8)
        plt.title(f"Velocity Over Time — {basefolder.name}", fontsize=12, pad=10)
        plt.xlabel("Time (s)", fontsize=10)
        plt.ylabel("Speed (deg/s)", fontsize=10)
        plt.grid(True, linestyle='--', alpha=0.5)
        plt.tight_layout()

        plot_save_path = plots_path / 'speed_overview.png'
        plt.savefig(plot_save_path, dpi=200)
        plt.close()
        print(f"  Saved plot: {plot_save_path}")

    except Exception as e:
        print(f"  Error processing {basefolder.name}: {e}")

print("\nAll files processed.")

C:\Users\Freitag\si_env\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.3.0)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(


Found 26 NIDQ files to process.

Processing: 52_naive
  Saved CSV : I:\Data\raw\52\52_naive\Meta\speed.csv
  Saved plot: I:\Data\raw\52\52_naive\plots\speed_overview.png

Processing: 52_recall
  Saved CSV : I:\Data\raw\52\52_recall\Meta\speed.csv
  Saved plot: I:\Data\raw\52\52_recall\plots\speed_overview.png

Processing: 51_naive
  Saved CSV : I:\Data\raw\51\51_naive\Meta\speed.csv
  Saved plot: I:\Data\raw\51\51_naive\plots\speed_overview.png

Processing: 51_recall
  Saved CSV : I:\Data\raw\51\51_recall\Meta\speed.csv
  Saved plot: I:\Data\raw\51\51_recall\plots\speed_overview.png

Processing: 17_recall
  Saved CSV : I:\Data\raw\17\17_recall\Meta\speed.csv
  Saved plot: I:\Data\raw\17\17_recall\plots\speed_overview.png

Processing: 17_naive
  Saved CSV : I:\Data\raw\17\17_naive\Meta\speed.csv
  Saved plot: I:\Data\raw\17\17_naive\plots\speed_overview.png

Processing: 7644_recall
  Saved CSV : I:\Data\raw\7644\7644_recall\Meta\speed.csv
  Saved plot: I:\Data\raw\7644\7644_recall\plots